# Silver: ERP customers
**Source:** `bronze.erp_cust_az12`  ->  **Target:** `silver.erp_customers`

**What this notebook does:**
- Remove extra spaces
- Fix customer ID (remove the `NAS` prefix so it matches the CRM)
- Birthdates in the future become empty
- Standardize gender
- Rename columns

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType

CATALOG = "workspace"

## Read the Bronze table

In [0]:
df = spark.table(f"{CATALOG}.bronze.erp_cust_az12")

## 1. Trim spaces

In [0]:
# Remove extra spaces from every text column
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

## 2. Fix customer ID
`NASAW00011000` -> `AW00011000`, so it matches the CRM customer number.

In [0]:
df = df.withColumn(
    "cid",
    F.when(F.col("cid").startswith("NAS"), F.substring("cid", 4, 100))
     .otherwise(F.col("cid"))
)

## 3. Birthdates
A birthdate in the future is impossible, so we make it empty.

In [0]:
df = df.withColumn("bdate", F.col("bdate").cast(DateType()))

df = df.withColumn(
    "bdate",
    F.when(F.col("bdate") > F.current_date(), None).otherwise(F.col("bdate"))
)

## 4. Standardize gender

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(F.col("gen")).isin("F", "FEMALE"), "Female")
     .when(F.upper(F.col("gen")).isin("M", "MALE"),   "Male")
     .otherwise("n/a")
)

## 5. Rename columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender",
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Write the Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.silver.erp_customers")

## Check it Quickly

In [0]:
result = spark.table(f"{CATALOG}.silver.erp_customers")
print("rows:", result.count())
result.groupBy("gender").count().display()